# 8.4 ?? ??? ?? ???? ?? ?? I : LightGBM ??

- [?? ??? ?? ???? ??](https://www.kaggle.com/c/porto-seguro-safe-driver-prediction)
- [??? ?? ?? ??](https://www.kaggle.com/xiaozhouwang/2nd-place-lightgbm-solution)

In [ ]:
import pandas as pd

# ??? ??
data_path = '/kaggle/input/porto-seguro-safe-driver-prediction/'

train = pd.read_csv(data_path + 'train.csv', index_col='id')
test = pd.read_csv(data_path + 'test.csv', index_col='id')
submission = pd.read_csv(data_path + 'sample_submission.csv', index_col='id')

## 8.4.1 ?? ?????

### ??? ???

In [ ]:
all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.drop('target', axis=1) # ??? ??

all_features = all_data.columns # ?? ??

### ??? ?? ?-? ???

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# ??? ??
cat_features = [feature for feature in all_features if 'cat' in feature] 

# ?-? ??? ??
onehot_encoder = OneHotEncoder()
encoded_cat_matrix = onehot_encoder.fit_transform(all_data[cat_features]) 

### ?? ?? ??

In [ ]:
# '??? ??? ??? ??'? ?? ??? ??
all_data['num_missing'] = (all_data==-1).sum(axis=1)

In [ ]:
# ??? ??, calc ??? ??? ??? ??
remaining_features = [feature for feature in all_features
                      if ('cat' not in feature and 'calc' not in feature)] 
# num_missing? remaining_features? ??
remaining_features.append('num_missing')

In [ ]:
# ??? ind? ??
ind_features = [feature for feature in all_features if 'ind' in feature]

is_first_feature = True
for ind_feature in ind_features:
    if is_first_feature:
        all_data['mix_ind'] = all_data[ind_feature].astype(str) + '_'
        is_first_feature = False
    else:
        all_data['mix_ind'] += all_data[ind_feature].astype(str) + '_'

In [ ]:
all_data['mix_ind']

In [ ]:
cat_count_features = []
for feature in cat_features+['mix_ind']:
    val_counts_dict = all_data[feature].value_counts().to_dict()
    all_data[f'{feature}_count'] = all_data[feature].apply(lambda x: 
                                                           val_counts_dict[x])
    cat_count_features.append(f'{feature}_count')

In [ ]:
cat_count_features

### ?? ?? ?? ??

In [ ]:
from scipy import sparse
# ?? ?? ???
drop_features = ['ps_ind_14', 'ps_ind_10_bin', 'ps_ind_11_bin', 
                 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_car_14']

# remaining_features, cat_count_features?? drop_features? ??? ???
all_data_remaining = all_data[remaining_features+cat_count_features].drop(drop_features, axis=1)

# ??? ???
all_data_sprs = sparse.hstack([sparse.csr_matrix(all_data_remaining),
                               encoded_cat_matrix],
                              format='csr')

### ??? ???

In [ ]:
num_train = len(train) # ?? ??? ??

# ?? ???? ??? ??? ???
X = all_data_sprs[:num_train]
X_test = all_data_sprs[num_train:]

y = train['target'].values

### ??? ???? ?? ??

In [ ]:
import numpy as np

def eval_gini(y_true, y_pred):
    # ???? ???? ??? ??? ?? (?? ??? ?? ??)
    assert y_true.shape == y_pred.shape

    n_samples = y_true.shape[0]                      # ??? ??
    L_mid = np.linspace(1 / n_samples, 1, n_samples) # ??? ?

    # 1) ???? ?? ????
    pred_order = y_true[y_pred.argsort()] # y_pred ????? y_true ? ??
    L_pred = np.cumsum(pred_order) / np.sum(pred_order) # ??? ??
    G_pred = np.sum(L_mid - L_pred)       # ?? ?? ?? ????

    # 2) ??? ??? ? ????
    true_order = y_true[y_true.argsort()] # y_true ????? y_true ? ??
    L_true = np.cumsum(true_order) / np.sum(true_order) # ??? ??
    G_true = np.sum(L_mid - L_true)       # ??? ??? ? ????

    # ???? ????
    return G_pred / G_true

In [ ]:
# LightGBM? gini() ??
def gini(preds, dtrain):
    labels = dtrain.get_label()
    return 'gini', eval_gini(labels, preds), True # ???

## 8.4.2 ??????? ???

### ???? ??

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# 8:2 ??? ?? ???, ?? ??? ?? (???? ??? ???)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, 
                                                      test_size=0.2, 
                                                      random_state=0)

# ???? ???? ????
bayes_dtrain = lgb.Dataset(X_train, y_train)
bayes_dvalid = lgb.Dataset(X_valid, y_valid)

### ??????? ?? ??

In [ ]:
# ???? ???? ?? ??????? ??
param_bounds = {'num_leaves': (30, 40),
                'lambda_l1': (0.7, 0.9),
                'lambda_l2': (0.9, 1),
                'feature_fraction': (0.6, 0.7),
                'bagging_fraction': (0.6, 0.9),
                'min_child_samples': (6, 10),
                'min_child_weight': (10, 40)}

# ?? ??? ???????
fixed_params = {'objective': 'binary',
                'learning_rate': 0.005,
                'bagging_freq': 1,
                'force_row_wise': True,
                'random_state': 1991}

### (???? ????) ???? ?? ?? ??

In [ ]:
def eval_function(num_leaves, lambda_l1, lambda_l2, feature_fraction,
                  bagging_fraction, min_child_samples, min_child_weight):
    '''?????? ????(????) ?? ??'''
    
    # ???? ???? ??? ??????? 
    params = {'num_leaves': int(round(num_leaves)),
              'lambda_l1': lambda_l1,
              'lambda_l2': lambda_l2,
              'feature_fraction': feature_fraction,
              'bagging_fraction': bagging_fraction,
              'min_child_samples': int(round(min_child_samples)),
              'min_child_weight': min_child_weight,
              'feature_pre_filter': False}
    # ??? ???????? ??
    params.update(fixed_params)
    
    print('???????:', params)    
    
    # LightGBM ?? ??
    lgb_model = lgb.train(params=params, 
                           train_set=bayes_dtrain,
                           num_boost_round=2500,
                           valid_sets=bayes_dvalid,
                           feval=gini,
                           early_stopping_rounds=300,
                           verbose_eval=False)
    # ?? ???? ?? ??
    preds = lgb_model.predict(X_valid) 
    # ???? ??
    gini_score = eval_gini(y_valid, preds)
    print(f'???? : {gini_score}\n')
    
    return gini_score

### ??? ??

In [ ]:
from bayes_opt import BayesianOptimization

# ???? ??? ?? ??
optimizer = BayesianOptimization(f=eval_function,      # ???? ?? ??
                                 pbounds=param_bounds, # ??????? ??
                                 random_state=0)

In [ ]:
# ???? ??? ??
optimizer.maximize(init_points=3, n_iter=6)

### ?? ??

In [ ]:
# ???? ??? ??? ? ???????
max_params = optimizer.max['params']
max_params

In [ ]:
# ??? ??????? ??
max_params['num_leaves'] = int(round(max_params['num_leaves']))
max_params['min_child_samples'] = int(round(max_params['min_child_samples']))

In [ ]:
# ?? ??? ??????? ??
max_params.update(fixed_params)

In [ ]:
max_params

## 8.4.3 ?? ?? ? ?? ??

In [ ]:
from sklearn.model_selection import StratifiedKFold

# ?? K ?? ?? ??? ??
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=1991)

# OOF ???? ??? ??? ?? ??? ???? ??? ??? ?? 1?? ??
oof_val_preds = np.zeros(X.shape[0]) 
# OOF ???? ??? ??? ??? ??? ???? ??? ??? ?? 1?? ??
oof_test_preds = np.zeros(X_test.shape[0]) 

# OOF ???? ?? ??, ??, ??
for idx, (train_idx, valid_idx) in enumerate(folds.split(X, y)):
    # ? ??? ???? ?? ??
    print('#'*40, f'?? {idx+1} / ?? {folds.n_splits}', '#'*40)
    
    # ??? ???, ??? ??? ??
    X_train, y_train = X[train_idx], y[train_idx] # ??? ???
    X_valid, y_valid = X[valid_idx], y[valid_idx] # ??? ???

    # LightGBM ?? ???? ??
    dtrain = lgb.Dataset(X_train, y_train) # LightGBM ?? ?? ????
    dvalid = lgb.Dataset(X_valid, y_valid) # LightGBM ?? ?? ????
                          
    # LightGBM ?? ??
    lgb_model = lgb.train(params=max_params,    # ?? ???????
                          train_set=dtrain,     # ?? ????
                          num_boost_round=2500, # ??? ?? ??
                          valid_sets=dvalid,    # ?? ??? ?? ????
                          feval=gini,           # ??? ????
                          early_stopping_rounds=300, # ???? ??
                          verbose_eval=100)     # 100???? ?? ??
    
    # ??? ???? ??? OOF ??
    oof_test_preds += lgb_model.predict(X_test)/folds.n_splits
    # ?? ?? ??? ?? ?? ??? ??? ?? 
    oof_val_preds[valid_idx] += lgb_model.predict(X_valid)
    
    # ?? ??? ????? ?? ??? ????
    gini_score = eval_gini(y_valid, oof_val_preds[valid_idx])
    print(f'?? {idx+1} ???? : {gini_score}\n')

In [ ]:
print('OOF ?? ??? ???? :', eval_gini(y, oof_val_preds))

## 8.4.4 ?? ? ?? ??

In [ ]:
submission['target'] = oof_test_preds
submission.to_csv('submission.csv')